# IAM Forms A-D — Top-5-Writer Crop Extraction (Demo)

Small, self-contained demonstration notebook. It:

1. Extracts `formsA-D.tgz` and `ascii.tgz`
2. Builds the true `form_id -> writer_id` map from `forms.txt` (no filename guessing)
3. Groups all form images by real writer ID
4. Picks the **5 writers with the most images**
5. Crops each form down to just the handwritten body (removes printed header/footer)
6. Saves everything into `sample_writers/writer_001/ ... writer_005/`
7. Shows a quick visual sanity check of the results

Run the cells top to bottom.


## 1. Configuration

Mounts Google Drive and points at the same folder used in the main project notebook. **Edit `DRIVE_DATA_DIR` below** if your `formsA-D.tgz` / `ascii.tgz` live somewhere else.

In [ ]:
import os
import tarfile
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

# ============================================================
# GOOGLE DRIVE MOUNT -- same pattern as the main project notebook.
# This demo is independent (doesn't reuse anything from that notebook's
# runtime), but it reads the SAME .tgz files from the SAME Drive folder,
# so it's mounted here too rather than assuming the files are already
# sitting in the local working directory.
# ============================================================
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_MOUNTED = True
except Exception as e:
    print(f"Drive mount not available in this environment ({e}). "
          f"If you're on Kaggle, use Kaggle's Add Data -> Google Drive integration, "
          f"or upload formsA-D.tgz / ascii.tgz as a Kaggle Dataset and point "
          f"DRIVE_DATA_DIR at that path instead.")
    DRIVE_MOUNTED = False

# --------------------------------------------------------------------------
# EDIT THIS to match the Drive folder used in the main project notebook
# (the one containing formsA-D.tgz and ascii.tgz)
DRIVE_DATA_DIR = "/content/drive/MyDrive/IAM_Handwriting"
# --------------------------------------------------------------------------

# ============================================================
# CONFIG
# ============================================================
FORMS_TGZ = os.path.join(DRIVE_DATA_DIR, "formsA-D.tgz")
ASCII_TGZ = os.path.join(DRIVE_DATA_DIR, "ascii.tgz")
EXTRACT_DIR = "iam_extracted"          # local scratch space, not on Drive
OUTPUT_DIR = "sample_writers"          # local output -- copy/download after running if needed
TOP_N_WRITERS = 5

FORMS_EXTRACT_DIR = os.path.join(EXTRACT_DIR, "formsA-D")
ASCII_EXTRACT_DIR = os.path.join(EXTRACT_DIR, "ascii")
os.makedirs(FORMS_EXTRACT_DIR, exist_ok=True)
os.makedirs(ASCII_EXTRACT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Config set.")
print(f"  DRIVE_DATA_DIR = {DRIVE_DATA_DIR}")
print(f"  FORMS_TGZ = {FORMS_TGZ}")
print(f"  ASCII_TGZ = {ASCII_TGZ}")
print(f"  OUTPUT_DIR = {OUTPUT_DIR}")


## 2. Extract `formsA-D.tgz` and `ascii.tgz`

In [ ]:

assert os.path.exists(FORMS_TGZ), (
    f"Not found: {FORMS_TGZ} -- check that DRIVE_DATA_DIR in the config cell "
    f"above points at the right Drive folder."
)
assert os.path.exists(ASCII_TGZ), (
    f"Not found: {ASCII_TGZ} -- check that DRIVE_DATA_DIR in the config cell "
    f"above points at the right Drive folder."
)

with tarfile.open(FORMS_TGZ, "r:gz") as tf_:
    tf_.extractall(FORMS_EXTRACT_DIR)
print(f"Extracted forms archive -> {FORMS_EXTRACT_DIR}")

with tarfile.open(ASCII_TGZ, "r:gz") as tf_:
    tf_.extractall(ASCII_EXTRACT_DIR)
print(f"Extracted ascii archive -> {ASCII_EXTRACT_DIR}")


## 3. Parse `forms.txt` — real writer IDs

`forms.txt` lives inside `ascii.tgz` and maps every `form_id` to its true `writer_id`.
We use **only** this mapping for writer identity — never the filename.


In [ ]:

forms_txt_candidates = list(Path(ASCII_EXTRACT_DIR).rglob("forms.txt"))
assert forms_txt_candidates, "forms.txt not found inside ascii.tgz -- check archive contents."
forms_txt_path = forms_txt_candidates[0]
print("Found forms.txt at:", forms_txt_path)

form_to_writer = {}
with open(forms_txt_path, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split()
        if len(parts) < 2:
            continue
        form_id, writer_id = parts[0], parts[1]
        form_to_writer[form_id] = writer_id

print(f"Parsed {len(form_to_writer)} form -> writer mappings from forms.txt")


## 4. Group form images by real writer ID

In [ ]:

image_paths = list(Path(FORMS_EXTRACT_DIR).rglob("*.png"))
if not image_paths:
    image_paths = list(Path(FORMS_EXTRACT_DIR).rglob("*.tif")) or \
                  list(Path(FORMS_EXTRACT_DIR).rglob("*.jpg"))
assert image_paths, f"No form images found under {FORMS_EXTRACT_DIR} -- check archive contents."

writer_to_images = {}
unmatched = 0
for p in image_paths:
    form_id = p.stem  # e.g. "a01-000u"
    writer_id = form_to_writer.get(form_id)
    if writer_id is None:
        unmatched += 1
        continue  # skip images with no forms.txt entry -- no filename guessing
    writer_to_images.setdefault(writer_id, []).append(p)

print(f"Total image files found: {len(image_paths)}")
print(f"Images matched to a writer via forms.txt: {sum(len(v) for v in writer_to_images.values())}")
print(f"Unmatched (excluded): {unmatched}")
print(f"Total distinct writers found: {len(writer_to_images)}")


## 5. Select the top 5 writers by image count

In [ ]:

top_writers = sorted(writer_to_images.items(), key=lambda kv: len(kv[1]), reverse=True)[:TOP_N_WRITERS]

print(f"Top {TOP_N_WRITERS} writers by image count:")
for writer_id, imgs in top_writers:
    print(f"  IAM writer {writer_id}: {len(imgs)} images")


## 6. Crop function — row-wise dark-pixel heuristic

IAM forms are ruled with two near-full-width horizontal lines: one separating the
printed header from the handwriting, and one separating the handwriting from the
printed footer / segmentation box. We find those lines from the fraction of dark
pixels in each row (a ruled line lights up almost the entire row width) and crop
strictly between them. If a line can't be found confidently, we fall back to a
fixed proportional crop.


In [ ]:

def crop_handwriting_region(img_u8, header_band=(0.05, 0.35), footer_band=(0.65, 0.98),
                             line_thresh=0.75, margin_px=15):
    h, w = img_u8.shape
    dark_frac = (img_u8 < 128).mean(axis=1)  # fraction of dark pixels per row

    def find_ruled_line(lo, hi):
        y0, y1 = int(lo * h), int(hi * h)
        band = dark_frac[y0:y1]
        if band.size == 0 or band.max() < line_thresh:
            return None
        return y0 + int(np.argmax(band))

    top = find_ruled_line(*header_band)
    bot = find_ruled_line(*footer_band)

    top = (top + margin_px) if top is not None else int(0.20 * h)
    bot = (bot - margin_px) if bot is not None else int(0.78 * h)
    top, bot = max(0, top), min(h, bot)
    if bot - top < 0.15 * h:  # safety net if line detection failed badly
        top, bot = int(0.20 * h), int(0.78 * h)
    return img_u8[top:bot, :]

print("crop_handwriting_region() defined.")


## 7. Crop and save into `sample_writers/`

```
sample_writers/
├── writer_001/
│   ├── form1_cropped.png
│   └── ...
├── writer_002/
...
```

Folders are numbered by rank (most images first); the IAM writer ID is printed
alongside each folder so results can be traced back to the source data.


In [ ]:

saved_summary = []  # (folder_name, iam_writer_id, n_saved, sample_output_path)

for rank, (writer_id, img_paths) in enumerate(top_writers, start=1):
    folder_name = f"writer_{rank:03d}"
    writer_folder = os.path.join(OUTPUT_DIR, folder_name)
    os.makedirs(writer_folder, exist_ok=True)

    n_saved = 0
    first_out_path = None
    for img_path in img_paths:
        img = np.array(Image.open(img_path).convert("L"))  # grayscale
        cropped = crop_handwriting_region(img)

        out_name = img_path.stem + "_cropped.png"
        out_path = os.path.join(writer_folder, out_name)
        Image.fromarray(cropped).save(out_path)
        if first_out_path is None:
            first_out_path = out_path
        n_saved += 1

    saved_summary.append((folder_name, writer_id, n_saved, first_out_path))
    print(f"{folder_name} (IAM writer {writer_id}): saved {n_saved} cropped images -> {writer_folder}")

print(f"\nDone. Output structure ready under '{OUTPUT_DIR}/'")


## 8. Visual sanity check

One cropped sample per writer, so the crop quality can be confirmed at a glance.

In [ ]:

fig, axes = plt.subplots(1, len(saved_summary), figsize=(4 * len(saved_summary), 5))
if len(saved_summary) == 1:
    axes = [axes]

for ax, (folder_name, writer_id, n_saved, sample_path) in zip(axes, saved_summary):
    img = Image.open(sample_path)
    ax.imshow(img, cmap="gray")
    ax.set_title(f"{folder_name}\n(IAM writer {writer_id}, {n_saved} imgs)", fontsize=10)
    ax.axis("off")

plt.suptitle("One cropped sample per top-5 writer", y=1.05)
plt.tight_layout()
plt.show()


## 9. Final folder structure

In [ ]:

print(f"{OUTPUT_DIR}/")
for folder_name, writer_id, n_saved, _ in saved_summary:
    print(f"├── {folder_name}/   ({n_saved} cropped images, IAM writer {writer_id})")
    folder_path = os.path.join(OUTPUT_DIR, folder_name)
    for fname in sorted(os.listdir(folder_path))[:3]:
        print(f"│   ├── {fname}")
    if n_saved > 3:
        print(f"│   └── ... ({n_saved - 3} more)")
